# 🤖 Bybit LSTM Model Training v2

**Улучшения:**
- Предсказывает % изменение цены (не абсолютную цену)
- Нормализация относительно текущей цены
- Работает для всех монет одинаково
- Добавлена валидация и тесты

In [ ]:
# 1. Установка зависимостей
!pip install -q tensorflow scikit-learn pandas numpy joblib matplotlib

In [ ]:
# 2. Загрузка данных
from google.colab import files
print("📂 Загрузите CSV файлы из ml_data/:")
print("   - klines_BTCUSDT_60.csv")
print("   - klines_ETHUSDT_60.csv")
print("   - klines_SOLUSDT_60.csv")
print("   - klines_BNBUSDT_60.csv")
print("   - klines_XRPUSDT_60.csv")
uploaded = files.upload()

In [ ]:
import os
import pandas as pd
import numpy as np
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from sklearn.preprocessing import StandardScaler
import joblib

print(f"TensorFlow: {tf.__version__}")
print(f"GPU: {tf.config.list_physical_devices('GPU')}")

In [ ]:
# 3. Загрузка и объединение данных
def load_data():
    all_data = []
    symbols = []
    
    for f in os.listdir('.'):
        if f.startswith('klines_') and f.endswith('.csv'):
            symbol = f.split('_')[1]
            print(f"📊 Loading {f}...")
            df = pd.read_csv(f)
            df['symbol'] = symbol
            all_data.append(df)
            symbols.append(symbol)
            print(f"   {len(df)} records, price range: ${df['close'].min():.2f} - ${df['close'].max():.2f}")
    
    combined = pd.concat(all_data, ignore_index=True)
    print(f"\n✅ Total: {len(combined)} records from {len(symbols)} symbols")
    return combined, symbols

df_raw, symbols = load_data()

In [ ]:
# 4. Подготовка фичей
# ВАЖНО: Нормализуем относительно текущей цены!

def prepare_features(df):
    """Подготовить нормализованные фичи"""
    df = df.copy()
    
    # Нормализуем ценовые данные относительно close
    df['open_norm'] = (df['open'] - df['close']) / df['close']
    df['high_norm'] = (df['high'] - df['close']) / df['close']
    df['low_norm'] = (df['low'] - df['close']) / df['close']
    
    # Нормализуем BB относительно close
    df['bb_upper_norm'] = (df['bb_upper'] - df['close']) / df['close']
    df['bb_lower_norm'] = (df['bb_lower'] - df['close']) / df['close']
    df['bb_width'] = (df['bb_upper'] - df['bb_lower']) / df['close']
    
    # Нормализуем SMA/EMA относительно close
    df['sma20_norm'] = (df['sma_20'] - df['close']) / df['close']
    df['sma50_norm'] = (df['sma_50'] - df['close']) / df['close']
    df['ema12_norm'] = (df['ema_12'] - df['close']) / df['close']
    df['ema26_norm'] = (df['ema_26'] - df['close']) / df['close']
    
    # RSI нормализуем 0-1
    df['rsi_norm'] = df['rsi'] / 100.0
    
    # MACD нормализуем относительно цены
    df['macd_norm'] = df['macd'] / df['close']
    df['macd_signal_norm'] = df['macd_signal'] / df['close']
    df['macd_hist_norm'] = df['macd_histogram'] / df['close']
    
    # ATR нормализуем относительно цены
    df['atr_norm'] = df['atr'] / df['close']
    
    # Stochastic уже 0-100, нормализуем 0-1
    df['stoch_k_norm'] = df['stoch_k'] / 100.0
    df['stoch_d_norm'] = df['stoch_d'] / 100.0
    
    # Volume - логарифм и нормализация
    df['volume_log'] = np.log1p(df['volume'])
    
    # Целевая переменная: % изменение цены через 1 час
    df['target'] = df.groupby('symbol')['close'].shift(-1)
    df['target_pct'] = (df['target'] - df['close']) / df['close']
    
    return df

df = prepare_features(df_raw)
print(f"\n📊 Target stats:")
print(f"   Mean: {df['target_pct'].mean()*100:.4f}%")
print(f"   Std:  {df['target_pct'].std()*100:.4f}%")
print(f"   Min:  {df['target_pct'].min()*100:.2f}%")
print(f"   Max:  {df['target_pct'].max()*100:.2f}%")

In [ ]:
# 5. Выбор фичей
FEATURES = [
    # Нормализованные цены
    'open_norm', 'high_norm', 'low_norm',
    # RSI
    'rsi_norm',
    # MACD
    'macd_norm', 'macd_signal_norm', 'macd_hist_norm',
    # Bollinger Bands
    'bb_upper_norm', 'bb_lower_norm', 'bb_width',
    # Moving Averages
    'sma20_norm', 'sma50_norm', 'ema12_norm', 'ema26_norm',
    # Volatility
    'atr_norm',
    # Stochastic
    'stoch_k_norm', 'stoch_d_norm',
    # Volume
    'volume_log',
    # Time features
    'hour_sin', 'hour_cos', 'day_sin', 'day_cos', 'month_sin', 'month_cos'
]

print(f"📋 Features: {len(FEATURES)}")
for i, f in enumerate(FEATURES):
    print(f"   {i+1}. {f}")

In [ ]:
# 6. Очистка данных
df_clean = df.dropna(subset=FEATURES + ['target_pct'])
print(f"📊 After cleaning: {len(df_clean)} records (removed {len(df) - len(df_clean)})")

# Убираем экстремальные выбросы (>10% изменение за час - аномалия)
df_clean = df_clean[abs(df_clean['target_pct']) < 0.10]
print(f"📊 After outlier removal: {len(df_clean)} records")

In [ ]:
# 7. Создание последовательностей
SEQUENCE_LENGTH = 60

def create_sequences(df, features, target_col, seq_len):
    X, y = [], []
    
    # Обрабатываем каждый символ отдельно
    for symbol in df['symbol'].unique():
        df_sym = df[df['symbol'] == symbol].sort_values('timestamp')
        
        X_data = df_sym[features].values
        y_data = df_sym[target_col].values
        
        for i in range(len(X_data) - seq_len):
            X.append(X_data[i:i+seq_len])
            y.append(y_data[i+seq_len-1])  # target для последнего элемента
    
    return np.array(X), np.array(y)

X, y = create_sequences(df_clean, FEATURES, 'target_pct', SEQUENCE_LENGTH)
print(f"\n📊 Sequences created:")
print(f"   X shape: {X.shape}")
print(f"   y shape: {y.shape}")

In [ ]:
# 8. Нормализация фичей
# Reshape для scaler
X_flat = X.reshape(-1, X.shape[-1])

scaler_X = StandardScaler()
X_scaled_flat = scaler_X.fit_transform(X_flat)
X_scaled = X_scaled_flat.reshape(X.shape)

# Target уже в процентах, просто масштабируем
scaler_y = StandardScaler()
y_scaled = scaler_y.fit_transform(y.reshape(-1, 1)).flatten()

print(f"✅ Data normalized")
print(f"   X mean: {X_scaled.mean():.4f}, std: {X_scaled.std():.4f}")
print(f"   y mean: {y_scaled.mean():.4f}, std: {y_scaled.std():.4f}")

In [ ]:
# 9. Train/Test split
split = int(len(X_scaled) * 0.8)
X_train, X_test = X_scaled[:split], X_scaled[split:]
y_train, y_test = y_scaled[:split], y_scaled[split:]

print(f"📊 Split:")
print(f"   Train: {len(X_train)}")
print(f"   Test:  {len(X_test)}")

In [ ]:
# 10. Модель
model = Sequential([
    Input(shape=(SEQUENCE_LENGTH, len(FEATURES))),
    
    LSTM(128, return_sequences=True),
    BatchNormalization(),
    Dropout(0.3),
    
    LSTM(64, return_sequences=False),
    BatchNormalization(),
    Dropout(0.3),
    
    Dense(32, activation='relu'),
    Dropout(0.2),
    
    Dense(1)  # Выход: % изменение
])

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='mse',
    metrics=['mae']
)

model.summary()

In [ ]:
# 11. Обучение
callbacks = [
    EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6)
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=100,
    batch_size=64,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
# 12. Визуализация обучения
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['loss'], label='Train')
axes[0].plot(history.history['val_loss'], label='Val')
axes[0].set_title('Loss')
axes[0].legend()

axes[1].plot(history.history['mae'], label='Train')
axes[1].plot(history.history['val_mae'], label='Val')
axes[1].set_title('MAE')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# 13. Оценка
train_loss, train_mae = model.evaluate(X_train, y_train, verbose=0)
test_loss, test_mae = model.evaluate(X_test, y_test, verbose=0)

print(f"📊 Results:")
print(f"   Train - Loss: {train_loss:.6f}, MAE: {train_mae:.6f}")
print(f"   Test  - Loss: {test_loss:.6f}, MAE: {test_mae:.6f}")

# Конвертируем MAE обратно в проценты
mae_pct = scaler_y.inverse_transform([[test_mae]])[0][0] * 100
print(f"\n   Test MAE в процентах: {abs(mae_pct):.4f}%")

In [ ]:
# 14. Тестовые предсказания
y_pred_scaled = model.predict(X_test, verbose=0)
y_pred = scaler_y.inverse_transform(y_pred_scaled).flatten()
y_true = scaler_y.inverse_transform(y_test.reshape(-1, 1)).flatten()

# Точность направления
direction_correct = np.sum((y_pred > 0) == (y_true > 0))
direction_accuracy = direction_correct / len(y_true) * 100

print(f"\n🎯 Direction Accuracy: {direction_accuracy:.1f}%")
print(f"   (Правильно предсказано направление движения)")

# Примеры предсказаний
print(f"\n📋 Sample predictions (% change):")
for i in range(10):
    idx = np.random.randint(0, len(y_pred))
    pred_pct = y_pred[idx] * 100
    true_pct = y_true[idx] * 100
    correct = "✅" if (pred_pct > 0) == (true_pct > 0) else "❌"
    print(f"   {correct} Pred: {pred_pct:+.3f}%, True: {true_pct:+.3f}%")

In [ ]:
# 15. Сохранение модели
model.save('bybit_lstm_model_v2.h5')
joblib.dump(scaler_X, 'scaler_X_v2.pkl')
joblib.dump(scaler_y, 'scaler_y_v2.pkl')

# Сохраняем список фичей
import json
metadata = {
    'version': 'v2',
    'created': datetime.now().isoformat(),
    'features': FEATURES,
    'sequence_length': SEQUENCE_LENGTH,
    'symbols': symbols,
    'total_samples': len(X),
    'train_samples': len(X_train),
    'test_samples': len(X_test),
    'epochs_trained': len(history.history['loss']),
    'test_loss': float(test_loss),
    'test_mae': float(test_mae),
    'direction_accuracy': float(direction_accuracy),
    'target_type': 'percentage_change',
    'notes': 'Predicts % price change in next hour'
}

with open('model_metadata_v2.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print("✅ Model saved!")
print("\n📁 Files to download:")
print("   - bybit_lstm_model_v2.h5")
print("   - scaler_X_v2.pkl")
print("   - scaler_y_v2.pkl")
print("   - model_metadata_v2.json")

In [ ]:
# 16. Скачать файлы
from google.colab import files

files.download('bybit_lstm_model_v2.h5')
files.download('scaler_X_v2.pkl')
files.download('scaler_y_v2.pkl')
files.download('model_metadata_v2.json')

print("\n✅ Готово! Загрузи файлы на сервер:")
print("   scp *.h5 *.pkl *.json root@88.210.10.145:/root/Bybit_Trader/ml_training/models/")